<a href="https://colab.research.google.com/github/fuger-tw/2026_DL_hws/blob/main/hw5_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import pickle

### 1. 將存好的模型和訓練權重讀回來用 (假設存放在SaveModel資料夾)

In [35]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
%cd '/content/drive/My Drive/Colab Notebooks'

/content/drive/My Drive/Colab Notebooks


In [37]:
from tensorflow.keras.models import model_from_json

In [38]:
model = model_from_json(open('SaveModel/imdb_model_architecture.json').read())
model.load_weights('SaveModel/imdb_model.weights.h5')

In [39]:
model.compile(loss='binary_crossentropy',
             optimizer='adam',
             metrics=['accuracy'])

In [40]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,411,713 (5.39 MB)

 Trainable params: 1,411,713 (5.39 MB)

 Non-trainable params: 0 (0.00 B)

### 2. 載入自行建立的tokenizer (假設存放在SaveModel資料夾)

In [41]:
import pickle
f = open('SaveModel/imdb_tokenizer_2000.pkl', 'rb')
tokens = pickle.load(f)
f.close()

In [42]:
from tensorflow.keras.preprocessing import sequence

In [43]:
# 情意分析標籤
SentimentDict = {
    1: '正面的',
    0: '負面的'
}

# 建立影評預測函數
def predict_review(input_text):

    # 1. 將文字轉換成數字序列
    input_seq = tokens.texts_to_sequences([input_text])

    # 2. 統一評論長度為 100
    pad_input_seq = sequence.pad_sequences(
        input_seq,
        maxlen=100
    )

    # 3. 使用模型進行預測
    predict_result = model.predict(
        pad_input_seq,
        verbose=0
    )

    # 4. 取得正面評論機率
    probability = float(predict_result[0][0])

    # 5. 判斷正面或負面
    sentiment = round(probability)

    result = SentimentDict[sentiment]

    # 6. 顯示分析結果
    print("影評內容：")
    print(input_text[:200] + "...")

    print("\n情意分析結果：", result)
    print("正面機率：", round(probability * 100, 2), "%")
    print("負面機率：", round((1 - probability) * 100, 2), "%")

    return {
        "結果": result,
        "正面機率": round(probability * 100, 2),
        "負面機率": round((1 - probability) * 100, 2)
    }

### 3. 輸入實際由IMDb取得的影評

In [44]:
# 第一則：Beauty and the Beast
# https://www.imdb.com/title/tt2771200/reviews/?utm_source=chatgpt.com
review1 = """
Sure, I'm a huge film snob who only likes artsy foreign films,
but that hasn't stopped me from loving Disney's Beauty & The Beast.
It's beautiful, breathtaking, warm and hilarious.
However, this film is so flat.
The directing was dull and uninteresting.
The CGI is horrid and the characters look lifeless.
Overall, this film is a complete bore.
"""

In [45]:
print("========== 第一則影評 ==========")
result1 = predict_review(review1)

========== 第一則影評 ==========
影評內容：

Sure, I'm a huge film snob who only likes artsy foreign films,
but that hasn't stopped me from loving Disney's Beauty & The Beast.
It's beautiful, breathtaking, warm and hilarious.
However, this film...

情意分析結果： 正面的
正面機率： 83.48 %
負面機率： 16.52 %


In [46]:
# 第二則：Mad Max: Fury Road
# https://www.imdb.com/title/tt1392190/reviews/?utm_source=chatgpt.com
review2 = """
It's hard to believe that the same talented director
who made Road Warrior had anything to do with this disaster.
This movie is the exact opposite.
There is absolutely nothing and no one to care about.
Tom Hardy is pathetic as Max.
There's no tension, no real struggle.
Everything George Miller got right with that masterpiece
he gets completely wrong here.
"""

In [47]:
print("\n========== 第二則影評 ==========")
result2 = predict_review(review2)


========== 第二則影評 ==========
影評內容：

It's hard to believe that the same talented director
who made Road Warrior had anything to do with this disaster.
This movie is the exact opposite.
There is absolutely nothing and no one to care abou...

情意分析結果： 正面的
正面機率： 89.19 %
負面機率： 10.81 %


In [48]:
# 第三則：The Shawshank Redemption
# https://www.imdb.com/title/tt0111161/reviews/?utm_source=chatgpt.com
review3 = """
The story is told in an unbeatable fluent storyflow
by a perfect cast.
It's a milestone of cinema with most intriguing characters.
Even though the twist is revealed after watching it,
I've watched it several times again.
"""

In [49]:
print("\n========== 第三則影評 ==========")
result3 = predict_review(review3)


========== 第三則影評 ==========
影評內容：

The story is told in an unbeatable fluent storyflow
by a perfect cast.
It's a milestone of cinema with most intriguing characters.
Even though the twist is revealed after watching it,
I've watched it...

情意分析結果： 正面的
正面機率： 99.99 %
負面機率： 0.01 %


#Gradio

In [50]:
!pip install gradio

In [51]:
import gradio as gr

In [52]:
# 建立 Web App 使用的預測函數
def gradio_predict(input_text):

    if not input_text.strip():
        return "請輸入電影評論！"

    # 呼叫自行建立的 predict_review 函數
    result = predict_review(input_text)

    # 整理輸出結果
    output = f"""
    情意分析結果：{result['結果']}

    正面評論機率：{result['正面機率']}%

    負面評論機率：{result['負面機率']}%
    """

    return output

In [53]:
# 建立 Gradio 介面
demo = gr.Interface(
    fn=gradio_predict,

    inputs=gr.Textbox(
        lines=10,
        label="請輸入電影評論",
        placeholder="Please enter your movie review here..."
    ),

    outputs=gr.Textbox(
        label="情意分析結果",
        lines=5
    ),

    title="IMDb 電影評論情意分析系統",

    description="""
    使用 LSTM 深度學習模型分析英文電影評論。

    輸入電影評論後，系統會判斷評論是正面或負面，
    並顯示兩種情緒的預測機率。
    """,

    examples=[
        [review1],
        [review2],
        [review3]
    ]
)

In [54]:
# 啟動 Web App
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6fdea0815ac1a79711.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
